In [1]:

import math
import random

# --------------------------------------------------
# OPEN VEHICLE ROUTING USING ANT COLONY OPTIMIZATION
# Based on Singh and Dhir (2014)
# --------------------------------------------------

random.seed(42)

# Example map coordinates for 21 points.
# Point 1 = Source, Point 2 = Destination.
# Points 3 to 21 = Intermediate stops.
# Replace these coordinates with real map coordinates.
points = [
    (0, 0),    # Point 1: Source
    (20, 10),  # Point 2: Destination
    (2, 4),    # Point 3
    (4, 1),    # Point 4
    (5, 7),    # Point 5
    (7, 3),    # Point 6
    (8, 9),    # Point 7
    (10, 5),   # Point 8
    (11, 1),   # Point 9
    (12, 8),   # Point 10
    (13, 4),   # Point 11
    (14, 10),  # Point 12
    (15, 2),   # Point 13
    (16, 6),   # Point 14
    (17, 12),  # Point 15
    (18, 4),   # Point 16
    (19, 8),   # Point 17
    (3, 10),   # Point 18
    (6, 12),   # Point 19
    (9, 12),   # Point 20
    (14, 13)   # Point 21
]

# ACO parameters from the paper
alpha = 2
beta = 6
rho = 0.5
Q = 2.7179
iterations = 1000

# Paper uses number of points as the number of ants
num_ants = len(points)

n = len(points)
source = 0
destination = 1

# Intermediate stops: points 3 through 21
intermediate = [
    i for i in range(n)
    if i != source and i != destination
]

# Calculate Euclidean distance between every pair of points
distance = [[0.0] * n for _ in range(n)]

for i in range(n):
    for j in range(n):
        x1, y1 = points[i]
        x2, y2 = points[j]

        distance[i][j] = math.sqrt(
            (x1 - x2) ** 2 + (y1 - y2) ** 2
        )

# Initialize pheromone trails
pheromone = [[1.0] * n for _ in range(n)]

best_route = None
best_distance = float("inf")


def route_length(route):
    """Calculate the length of an open route."""
    total = 0.0

    for i in range(len(route) - 1):
        total += distance[route[i]][route[i + 1]]

    return total


def construct_route():
    """Build a route from source to destination."""

    route = [source]
    unvisited = intermediate.copy()

    while unvisited:
        current = route[-1]
        weights = []

        # Calculate ACO transition probabilities
        for city in unvisited:
            tau = pheromone[current][city] ** alpha
            eta = (1.0 / distance[current][city]) ** beta

            weights.append(tau * eta)

        total_weight = sum(weights)

        if total_weight == 0:
            next_city = random.choice(unvisited)
        else:
            probabilities = [
                weight / total_weight for weight in weights
            ]

            next_city = random.choices(
                unvisited,
                weights=probabilities,
                k=1
            )[0]

        route.append(next_city)
        unvisited.remove(next_city)

    # Fixed destination: do not return to source
    route.append(destination)

    return route


# Main ACO algorithm
for iteration in range(iterations):

    # Accumulate pheromone deposits for this iteration
    delta_pheromone = [
        [0.0] * n for _ in range(n)
    ]

    for ant in range(num_ants):

        # Construct an open route
        route = construct_route()
        length = route_length(route)

        # Save the shortest route found
        if length < best_distance:
            best_distance = length
            best_route = route.copy()

        # Deposit pheromone on route edges
        deposit = Q / length

        for i in range(len(route) - 1):
            a = route[i]
            b = route[i + 1]

            delta_pheromone[a][b] += deposit
            delta_pheromone[b][a] += deposit

    # Pheromone evaporation and reinforcement
    for i in range(n):
        for j in range(n):
            pheromone[i][j] = (
                rho * pheromone[i][j]
                + delta_pheromone[i][j]
            )

    # Display progress
    if (iteration + 1) % 100 == 0:
        print(
            f"Iteration {iteration + 1}: "
            f"Best distance = {best_distance:.4f}"
        )


# --------------------------------------------------
# DISPLAY FINAL RESULTS
# --------------------------------------------------

print("\n========== FINAL RESULTS ==========")

print("Algorithm: ACO with Open Tour")
print("Total points:", n)
print("Source point:", source + 1)
print("Destination point:", destination + 1)
print("Number of ants:", num_ants)
print("Iterations:", iterations)

print("\nBest Open Route:")
print(" -> ".join(str(city + 1) for city in best_route))

print(f"\nMinimum route distance: {best_distance:.4f}")

print("\nRoute verification:")
print("Starts at source:", best_route[0] == source)
print("Ends at destination:", best_route[-1] == destination)
print(
    "Visits every point exactly once:",
    len(best_route) == n and len(set(best_route)) == n
)
print("Returns to source:", best_route[-1] == source)


Iteration 100: Best distance = 74.0958
Iteration 200: Best distance = 74.0958
Iteration 300: Best distance = 74.0958
Iteration 400: Best distance = 74.0958
Iteration 500: Best distance = 74.0958
Iteration 600: Best distance = 74.0958
Iteration 700: Best distance = 74.0958
Iteration 800: Best distance = 74.0958
Iteration 900: Best distance = 74.0958
Iteration 1000: Best distance = 74.0958

========== FINAL RESULTS ==========
Algorithm: ACO with Open Tour
Total points: 21
Source point: 1
Destination point: 2
Number of ants: 21
Iterations: 1000

Best Open Route:
1 -> 3 -> 4 -> 6 -> 9 -> 13 -> 16 -> 14 -> 11 -> 8 -> 7 -> 20 -> 19 -> 18 -> 5 -> 10 -> 12 -> 21 -> 15 -> 17 -> 2

Minimum route distance: 74.0958

Route verification:
Starts at source: True
Ends at destination: True
Visits every point exactly once: True
Returns to source: False
